# Bildungsabschluss als zusätzliches Merkmal

Wir prüfen `highest_education` zunächst beschreibend am Stichtag 27.
Danach vergleichen wir das bisherige Wochenmodell mit derselben
Modellvariante plus Bildungsabschluss. Beide Varianten nutzen dieselben
personengetrennten Folds. Die Präsentation 2014J bleibt bei diesem
Merkmalsvergleich außen vor.

Unterschiedliche Ergebnisanteile zwischen Bildungsgruppen beschreiben
Zusammenhänge; sie belegen keine Ursache.

In [1]:
from pathlib import Path
import pandas as pd

# Projektordner finden, auch wenn das Notebook in einem Unterordner liegt.
projektordner = Path.cwd()
if projektordner.name.lower() == "notebooks":
    projektordner = projektordner.parent

schluessel = ["code_module", "code_presentation", "id_student"]
stichtage = [6, 27, 55, 111]

wochen_datei = (
    projektordner / "data" / "processed"
    / "oulad_wochenmodellbasis_tag6_bis111.csv"
)
info_datei = projektordner / "Daten OULAD" / "studentInfo.csv"

assert wochen_datei.exists(), f"Wochenbasis nicht gefunden: {wochen_datei}"
assert info_datei.exists(), f"studentInfo nicht gefunden: {info_datei}"

# Nur vier Beispielwochen laden; jede steht für eine andere Kursphase.
wochen = pd.read_csv(wochen_datei)
wochen = wochen.loc[wochen["stichtag"].isin(stichtage)].copy()

# Der Abschluss gehört zum Kurseintrag und wird über dessen Schlüssel ergänzt.
bildung = pd.read_csv(
    info_datei,
    usecols=schluessel + ["highest_education"],
)
assert not bildung.duplicated(schluessel).any()

wochen = wochen.merge(
    bildung,
    on=schluessel,
    how="left",
    validate="many_to_one",
)

# Vor der Modellierung prüfen wir, ob Einträge keinen Abschluss erhalten haben.
fehlend = wochen["highest_education"].isna() | wochen["highest_education"].eq("?")
print("Datenstände in vier Wochen:", len(wochen))
print("Ohne lesbaren Bildungsabschluss:", int(fehlend.sum()))
print("\nAusprägungen:")
print(wochen["highest_education"].value_counts(dropna=False))

# Eine fehlende Angabe bleibt als eigene Kategorie erkennbar.
wochen["highest_education"] = wochen["highest_education"].mask(
    fehlend, "Unbekannt"
)

# Nur Tag 27 beschreiben: Sonst würde derselbe Kurseintrag mehrfach zählen.
tag27 = wochen.loc[wochen["stichtag"].eq(27)]
uebersicht_bildung = (
    tag27.groupby("highest_education", dropna=False)
    .agg(
        kurseintraege=("id_student", "size"),
        abbruch_oder_fail=("abbruch_oder_fail", "sum"),
    )
)
uebersicht_bildung["anteil_prozent"] = (
    100
    * uebersicht_bildung["abbruch_oder_fail"]
    / uebersicht_bildung["kurseintraege"]
).round(1)

print("\nErgebnisse am Stichtag 27:")
display(uebersicht_bildung)

Datenstände in vier Wochen: 107699
Ohne lesbaren Bildungsabschluss: 0

Ausprägungen:
highest_education
A Level or Equivalent          47241
Lower Than A Level             42072
HE Qualification               16228
Post Graduate Qualification     1123
No Formal quals                 1035
Name: count, dtype: int64

Ergebnisse am Stichtag 27:


,kurseintraege,abbruch_oder_fail,anteil_prozent
highest_education,,,
A Level or Equivalent,12004,4700,39.2
HE Qualification,4136,1480,35.8
Lower Than A Level,10748,5638,52.5
No Formal quals,260,157,60.4
Post Graduate Qualification,287,82,28.6


In [2]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Das sind die bisher genutzten Wochenmerkmale. Der Schlüssel id_student
# und das spätere Ergebnis sind ausdrücklich keine Eingaben des Modells.
merkmale_bisher = [
    "code_module",
    "code_presentation",
    "anmeldetag",
    "assessments_faellig",
    "abgaben",
    "banked_faellige",
    "assessments_fehlend",
    "klicks_bis_stichtag",
    "klicks_letzte_7_tage",
]

varianten = {
    "Bisher": merkmale_bisher,
    "Mit Bildungsabschluss": merkmale_bisher + ["highest_education"],
}


def baue_modell(merkmale):
    """Erzeugt für beide Varianten dieselbe Modellpipeline."""
    kategorien = [
        spalte for spalte in merkmale
        if spalte in ["code_module", "code_presentation", "highest_education"]
    ]
    zahlen = [spalte for spalte in merkmale if spalte not in kategorien]

    vorbereitung = ColumnTransformer([
        (
            "kategorien",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            kategorien,
        ),
        ("zahlen", StandardScaler(), zahlen),
    ])

    return Pipeline([
        ("vorbereitung", vorbereitung),
        ("modell", HistGradientBoostingClassifier(random_state=42)),
    ])


def hinweis_kennzahlen(validierung, risiko, anteil=0.20):
    """Wählt je Kurspräsentation die obersten 20 % der Risikowerte."""
    auswahl = validierung[
        ["code_module", "code_presentation", "abbruch_oder_fail"]
    ].copy()
    auswahl["risiko"] = risiko

    gruppen = ["code_module", "code_presentation"]
    rang = auswahl.groupby(gruppen)["risiko"].rank(
        ascending=False, method="first"
    )
    gruppengroesse = auswahl.groupby(gruppen)["risiko"].transform("size")
    hinweis = rang.le(np.ceil(gruppengroesse * anteil))

    treffer = auswahl.loc[hinweis, "abbruch_oder_fail"].sum()
    return (
        treffer / auswahl["abbruch_oder_fail"].sum(),
        treffer / hinweis.sum(),
    )

In [3]:
ergebnisse_bildung = []

for stichtag in stichtage:
    # 2014J nicht für die Auswahl des zusätzlichen Merkmals verwenden.
    daten = wochen.loc[
        wochen["stichtag"].eq(stichtag)
        & wochen["code_presentation"].ne("2014J")
    ].reset_index(drop=True)

    ziel = daten["abbruch_oder_fail"].astype(int)
    falten = StratifiedGroupKFold(
        n_splits=3, shuffle=True, random_state=42
    )

    for fold, (train_idx, valid_idx) in enumerate(
        falten.split(daten, ziel, groups=daten["id_student"]),
        start=1,
    ):
        training = daten.iloc[train_idx]
        validierung = daten.iloc[valid_idx]

        # Beide Varianten sehen exakt dieselben Trainings- und Prüffälle.
        for name, merkmale in varianten.items():
            modell = baue_modell(merkmale)
            modell.fit(
                training[merkmale],
                training["abbruch_oder_fail"].astype(int),
            )
            risiko = modell.predict_proba(
                validierung[merkmale]
            )[:, 1]

            recall, trefferquote = hinweis_kennzahlen(
                validierung, risiko
            )

            ergebnisse_bildung.append({
                "stichtag": stichtag,
                "fold": fold,
                "variante": name,
                "AP": average_precision_score(
                    validierung["abbruch_oder_fail"], risiko
                ),
                "recall_20": recall,
                "trefferquote_20": trefferquote,
            })

    print(f"Fertig: Tag {stichtag}")

vergleich_bildung = pd.DataFrame(ergebnisse_bildung)
display(
    vergleich_bildung
    .groupby(["stichtag", "variante"])
    .agg(
        AP=("AP", "mean"),
        recall_20=("recall_20", "mean"),
        trefferquote_20=("trefferquote_20", "mean"),
    )
    .round(3)
)

Fertig: Tag 6
Fertig: Tag 27
Fertig: Tag 55
Fertig: Tag 111


AP  recall_20  trefferquote_20
stichtag variante                                                
6        Bisher                 0.644      0.286            0.679
         Mit Bildungsabschluss  0.669      0.294            0.697
27       Bisher                 0.717      0.333            0.759
         Mit Bildungsabschluss  0.733      0.343            0.782
55       Bisher                 0.763      0.383            0.833
         Mit Bildungsabschluss  0.773      0.387            0.841
111      Bisher                 0.825      0.454            0.909
         Mit Bildungsabschluss  0.830      0.458            0.916

In [4]:
# Vergleicht die beiden Varianten innerhalb desselben Stichtags
# und desselben Folds. Positive Werte sprechen für den Zusatz
# des Bildungsabschlusses.
paare = vergleich_bildung.pivot(
    index=["stichtag", "fold"],
    columns="variante",
    values=["AP", "recall_20", "trefferquote_20"],
)

differenzen = pd.DataFrame(index=paare.index)

for metrik in ["AP", "recall_20", "trefferquote_20"]:
    differenzen[metrik] = (
        paare[(metrik, "Mit Bildungsabschluss")]
        - paare[(metrik, "Bisher")]
    )

print("Unterschied je Fold:")
display(differenzen.round(3))

print("Mittlerer, kleinster und größter Unterschied je Stichtag:")
display(
    differenzen.groupby(level="stichtag")
    .agg(["mean", "min", "max"])
    .round(3)
)

Unterschied je Fold:


AP  recall_20  trefferquote_20
stichtag fold                                   
6        1     0.023      0.005            0.011
         2     0.030      0.013            0.031
         3     0.023      0.005            0.011
27       1     0.018      0.007            0.016
         2     0.011      0.009            0.021
         3     0.019      0.014            0.031
55       1     0.014      0.010            0.022
         2     0.008      0.003            0.007
         3     0.006     -0.001           -0.003
111      1     0.003      0.000            0.000
         2     0.005      0.007            0.014
         3     0.007      0.003            0.006

Mittlerer, kleinster und größter Unterschied je Stichtag:


AP               recall_20               trefferquote_20         \
           mean    min    max      mean    min    max            mean    min   
stichtag                                                                       
6         0.025  0.023  0.030     0.007  0.005  0.013           0.018  0.011   
27        0.016  0.011  0.019     0.010  0.007  0.014           0.023  0.016   
55        0.010  0.006  0.014     0.004 -0.001  0.010           0.009 -0.003   
111       0.005  0.003  0.007     0.003  0.000  0.007           0.007  0.000   

                 
            max  
stichtag         
6         0.031  
27        0.031  
55        0.022  
111       0.014

### Hinweise nach Bildungsabschluss

Wir vergleichen für Tag 6 und 27, wie häufig jede Bildungsgruppe
einen Hinweis erhält und welcher Anteil der späteren Fälle innerhalb
der Gruppe erkannt wird. Die Auswahl erfolgt wie zuvor innerhalb jeder
Kurspräsentation. Kleine Gruppen werden vorsichtig interpretiert.

In [5]:
gruppen_ergebnisse = []

for stichtag in [6, 27]:
    daten = wochen.loc[
        wochen["stichtag"].eq(stichtag)
        & wochen["code_presentation"].ne("2014J")
    ].reset_index(drop=True)

    ziel = daten["abbruch_oder_fail"].astype(int)
    falten = StratifiedGroupKFold(
        n_splits=3, shuffle=True, random_state=42
    )

    for train_idx, valid_idx in falten.split(
        daten, ziel, groups=daten["id_student"]
    ):
        training = daten.iloc[train_idx]
        validierung = daten.iloc[valid_idx]

        for name, merkmale in varianten.items():
            modell = baue_modell(merkmale)
            modell.fit(
                training[merkmale],
                training["abbruch_oder_fail"].astype(int),
            )
            risiko = modell.predict_proba(
                validierung[merkmale]
            )[:, 1]

            # Pro Kurspräsentation erhalten die obersten 20 % einen Hinweis.
            auswahl = validierung[
                ["code_module", "code_presentation"]
            ].copy()
            auswahl["risiko"] = risiko

            kurs = ["code_module", "code_presentation"]
            rang = auswahl.groupby(kurs)["risiko"].rank(
                ascending=False, method="first"
            )
            kursgroesse = auswahl.groupby(kurs)["risiko"].transform(
                "size"
            )
            hinweis = rang.le(np.ceil(kursgroesse * 0.20))

            # Nur für die Auswertung: tatsächliches Ergebnis und
            # Bildungsabschluss zu den vorhergesagten Hinweisen legen.
            teil = validierung[
                ["highest_education", "abbruch_oder_fail"]
            ].copy()
            teil["hinweis"] = hinweis.to_numpy()
            teil["treffer"] = (
                teil["hinweis"] & teil["abbruch_oder_fail"]
            )
            teil["stichtag"] = stichtag
            teil["variante"] = name
            gruppen_ergebnisse.append(teil)

    print(f"Fertig: Tag {stichtag}")

gruppen_daten = pd.concat(gruppen_ergebnisse, ignore_index=True)

gruppen_vergleich = (
    gruppen_daten
    .groupby(["stichtag", "variante", "highest_education"])
    .agg(
        kurseintraege=("abbruch_oder_fail", "size"),
        spaetere_faelle=("abbruch_oder_fail", "sum"),
        hinweise=("hinweis", "sum"),
        treffer=("treffer", "sum"),
    )
)

gruppen_vergleich["hinweisanteil_%"] = (
    100 * gruppen_vergleich["hinweise"]
    / gruppen_vergleich["kurseintraege"]
).round(1)
gruppen_vergleich["recall_innerhalb_gruppe_%"] = (
    100 * gruppen_vergleich["treffer"]
    / gruppen_vergleich["spaetere_faelle"]
).round(1)
gruppen_vergleich["trefferquote_%"] = (
    100 * gruppen_vergleich["treffer"]
    / gruppen_vergleich["hinweise"].replace(0, np.nan)
).round(1)

display(gruppen_vergleich)

Fertig: Tag 6
Fertig: Tag 27


kurseintraege  \
stichtag variante              highest_education                            
6        Bisher                A Level or Equivalent                 8172   
                               HE Qualification                      2661   
                               Lower Than A Level                    7713   
                               No Formal quals                        205   
                               Post Graduate Qualification            183   
         Mit Bildungsabschluss A Level or Equivalent                 8172   
                               HE Qualification                      2661   
                               Lower Than A Level                    7713   
                               No Formal quals                        205   
                               Post Graduate Qualification            183   
27       Bisher                A Level or Equivalent                 7949   
                               HE Qualification                      2588   
                               Lower Than A Level                    7402   
                               No Formal quals                        193   
                               Post Graduate Qualification            177   
         Mit Bildungsabschluss A Level or Equivalent                 7949   
                               HE Qualification                      2588   
                               Lower Than A Level                    7402   
                               No Formal quals                        193   
                               Post Graduate Qualification            177   

                                                            spaetere_faelle  \
stichtag variante              highest_education                              
6        Bisher                A Level or Equivalent                   3432   
                               HE Qualification                        1041   
                               Lower Than A Level                      4361   
                               No Formal quals                          133   
                               Post Graduate Qualification               58   
         Mit Bildungsabschluss A Level or Equivalent                   3432   
                               HE Qualification                        1041   
                               Lower Than A Level                      4361   
                               No Formal quals                          133   
                               Post Graduate Qualification               58   
27       Bisher                A Level or Equivalent                   3200   
                               HE Qualification                         966   
                               Lower Than A Level                      4047   
                               No Formal quals                          121   
                               Post Graduate Qualification               52   
         Mit Bildungsabschluss A Level or Equivalent                   3200   
                               HE Qualification                         966   
                               Lower Than A Level                      4047   
                               No Formal quals                          121   
                               Post Graduate Qualification               52   

                                                            hinweise  treffer  \
stichtag variante              highest_education                                
6        Bisher                A Level or Equivalent            1629     1026   
                               HE Qualification                  470      272   
                               Lower Than A Level               1626     1232   
                               No Formal quals                    46       36   
                               Post Graduate Qualification        33       17   
         Mit Bildungsabschluss A Level or Equivalent       